# mini-beatrix-2s-control, continued — the softmax twin from its last clean checkpoint, with guards

**What this is.** `mini-beatrix-2s-control` is the pure-softmax twin of the full-splat `mini-beatrix-2s` craft (d1024 × L20,
ctx 4096, 16 heads, byte-trigram; the same banks, head, optimizer split and curriculum). Its first run destabilized: the
pre-clip gradient norm crossed the clip (1.0) at step 17,600 inside the fineweb phase, stayed above it through the curriculum
stages, and the held-out loss rose from 1.08 (step 16,000) to 1.57 (step 22,000). The shipped checkpoints before that point
are intact: `checkpoints/step_00014000` and `step_00016000` on the training repo.

**What this notebook does.** It continues the twin from `step_00016000` (1,600 steps before the first clip exceedance) as a
NEW craft — its own name, its own prefix on the training repo, the original run's record untouched — with the guards the
literature and the first run point at:

| arm (`CRAFT`) | precision | attention | QK-norm | micro-batch × accumulation |
|---|---|---|---|---|
| `mini-beatrix-2s-control-bf16` | bf16 autocast, TF32 on (the first run's setting) | as before | off | 16 × 4 |
| `mini-beatrix-2s-control-fp32` | full fp32: autocast off, TF32 off | fp32 | off | 4 × 16 |
| `mini-beatrix-2s-control-attn` | bf16 autocast, TF32 on | the attention blocks in fp32 | off | 8 × 8 |
| `mini-beatrix-2s-control-fix` | bf16 autocast, TF32 on | the attention blocks in fp32 | on | 8 × 8 |
| `mini-beatrix-2s-control-fp32-fix` | full fp32 | fp32 | on | 4 × 16 |
| `mini-beatrix-2s-control-attn-fp16` | bf16 autocast, TF32 on | fp16 flash inside the fp32 attention block: 10-bit inputs and P, fp32 softmax, flash pace | off | 8 × 8 |
| `mini-beatrix-2s-control-fix-fp16` | bf16 autocast, TF32 on | the same | on | 8 × 8 |
| `mini-beatrix-2s-control-bf16-qk` | bf16 autocast, TF32 on | as before (bf16 flash) | on | 16 × 4 |
| `mini-beatrix-2s-control-fix-fp16-c10k` | bf16 autocast, TF32 on | fp16 flash in the fp32 block, CONTINUED from the `-attn-fp16-r10k` arm's own checkpoint at `START_STEP` (the stop rule's fallback) | on | 8 × 8 |

Every arm trains the recipe's 262,144-token step; the fp32 forms hold more activation memory at the same tokens, so they run
smaller micro-batches (the 95 GB card ran out of memory at 16 × 4 with fp32 attention). The `-bf16` arm is the control: the
restart alone (fresh optimizer states) must be able to fail the way the first run did. The arms without QK-norm start from the
checkpoint's exact function; the QK-norm arms pay an insertion cost of about 0.9 in loss (measured on the twin: the norm removes
the sink positions' scale, which the trained model relies on, and no gain can give it back), which training must recover first
and which confounds their early gradient norms. C2 prints that cost on a real batch before anything trains.

**The start.** The trainer performs a WEIGHTS-ONLY start the first time a craft with an `init_from` spec has no record on the
hub: the source checkpoint (bf16 on the hub) is upcast into the fp32 masters; the optimizer states are fresh (the first run
stored none between its phase boundaries); the curriculum cursor is set from the step (warmup done, fineweb_main active at
3.894B of 5B tokens, then S0-S8 and the two anneal phases, chronological); the fineweb stream opens at a seed offset (a fresh
shuffle — the first run consumed the shuffle head; a recorded data-order discontinuity); with QK-norm on, its gains are set
from the trained q/k scales on one fineweb batch so the norm enters near-identity (a boundary write; the provenance lands in the
manifest). Every later session resumes from this craft's own `resume/latest.pt` exactly as before.

**QK-norm** (Dehghani et al. 2023, arXiv 2302.05442; Wortsman et al. 2023, arXiv 2309.14322): per-head RMS normalization of
q and k over the head dimension with learned per-head per-channel gains. **fp32 attention**: under bf16 autocast the attention
block (projections, logits, softmax, output) runs with autocast disabled.

Target hardware: RTX PRO 6000 Blackwell 96 GB (the first run's card: 3.68 s a step in bf16) or an H100 80 GB. The full-fp32
attention arms (`-attn`, `-fix`) run the memory-efficient fp32 kernel (7.2 s a step on an H100; torch's fused kernel measured
the same there); the `-fp16` arms run fp16 flash inside the fp32 attention block (10-bit inputs and P, fp32 softmax, a scaled
backward) at about the control's pace. C2 prints the measured pace and the peak memory before anything trains; C3 names the
attention kernel in its first lines.

In [ ]:
# C1 — pinned install. RESTART RUNTIME if the version line below changes.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%pip install -q "datasets>=5.0.0" "geolip-alephllm[train] @ git+https://github.com/AbstractEyes/alephllm@v0.10.13"
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 10, 13), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')

from geolip.alephllm.presets import PRESETS, CONTROL_RESUME_ARMS, make_control_resume_preset

# THE ARM: one of CONTROL_RESUME_ARMS (see the table above). Each arm is its
# own craft name = its own prefix on the training repo. `-bf16` is the control
# the others are read against; `-attn` = full fp32 attention (7.2 s a step on an
# H100); `-attn-fp16` = 10-bit attention inputs with the fp32 softmax at flash
# pace; `-bf16-qk` = QK-norm on the first run's own attention; `-fix-fp16-c10k`
# = the fp16 arm from 10,000 CONTINUED from its own checkpoint at START_STEP with
# QK-norm added (the stop rule's fallback). Every arm but the QK-norm ones starts
# from the checkpoint's exact function.
CRAFT = 'mini-beatrix-2s-control-attn'
START_STEP = 16000            # the registered arms start here; 14000 is the other clean checkpoint

def craft(name, start):
    '''The registered preset at 16,000; any other clean checkpoint builds the same arm under a derived name.'''
    assert name in CONTROL_RESUME_ARMS, f'{name!r} is not one of {sorted(CONTROL_RESUME_ARMS)}'
    if start == 16000:
        return PRESETS[name]
    return make_control_resume_preset(f'{name}-r{start // 1000}k', start_step=start, **CONTROL_RESUME_ARMS[name])

print(craft(CRAFT, START_STEP).model.name)

In [ ]:
# C2 (P) — PREFLIGHT: build, the switches, the step-1 read on the source weights, bench, VRAM gate. No training, no hub writes.
import time, torch
from google.colab import userdata
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data.tokenizers import build_tokenizer
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train.precision import autocast

HF_TOKEN = userdata.get('HF_TOKEN')
p = craft(CRAFT, START_STEP); cfg, tc, src = p.model, p.train, p.init_from
print(f"{cfg.name}: precision {tc.precision} | qk_norm {cfg.qk_norm!r} | attn_fp32 {cfg.attn_fp32} | kernel {cfg.attn_kernel!r} | "
      f"hub layers {list(cfg.hub_layers)} | from {src['prefix']}/{src['path']} (step {src['step']:,}, seed offset {src['seed_offset']})")
assert cfg.hub_layers == () and tc.head_addr_frozen is False, 'the twin: no hubs, the born-null head unfrozen'
tf32 = tc.precision != 'fp32'
torch.backends.cuda.matmul.allow_tf32 = tf32; torch.backends.cudnn.allow_tf32 = tf32
torch.manual_seed(tc.seed)
model = AlephLM(cfg).cuda()
ck = hf_hub_download(src['repo'], f"{src['prefix']}/{src['path']}", token=HF_TOKEN)
sd = {k: v.float() for k, v in load_file(ck).items()}
res = model.load_state_dict(sd, strict=False)
print(f"{model.param_count()/1e6:.1f}M params | guard params added {len(res.missing_keys)} | unexpected keys {len(res.unexpected_keys)}")
assert not res.unexpected_keys and all(k.endswith(('q_gain', 'k_gain')) for k in res.missing_keys), 'the checkpoint does not fit the craft'

# THE STEP-1 READ: the same weights, one real fineweb batch (a throwaway stream), the loss under the three precisions —
# the deltas are the rounding; the trainer's held-out gauge is the real series.
tok = build_tokenizer(cfg.tokenizer)
x = build_stream('fineweb-edu', tok, cfg.context, 2, seed=tc.seed + 99).next_batch().cuda()
model.eval()
attn = [b.attn for b in model.blocks]

def read(tag):
    with torch.no_grad():
        for a in attn: a.attn_fp32 = False
        with torch.autocast('cuda', dtype=torch.bfloat16):
            l_bf16 = float(model(x[:, :-1], targets=x[:, 1:]).loss)
        for a in attn: a.attn_fp32 = True
        with torch.autocast('cuda', dtype=torch.bfloat16):
            l_mix = float(model(x[:, :-1], targets=x[:, 1:]).loss)
        for a in attn: a.attn_fp32 = cfg.attn_fp32
        l_fp32 = float(model(x[:, :-1], targets=x[:, 1:]).loss)
    print(f"step-1 read ({tag}): bf16 {l_bf16:.5f} | bf16 + fp32 attention {l_mix:.5f} | fp32 {l_fp32:.5f} (TF32 {tf32}); "
          f"bf16 - fp32 {l_bf16 - l_fp32:+.5f}, fp32 attention - fp32 {l_mix - l_fp32:+.5f}")
    return l_fp32

for a in attn: a.qk_norm = ''                       # the guard off = the first run's attention: the precision read on the trained function
l_raw = read('guard off' if cfg.qk_norm else 'the arm')
if cfg.qk_norm:
    for a in attn: a.qk_norm = cfg.qk_norm
    recs = model.install_qk_gains(x[:, :-1])        # the boundary write, rehearsed (the trainer does its own at the start)
    l_inst = read('gains installed')
    print(f"QK-norm insertion on this batch: guard off {l_raw:.5f} -> gains installed {l_inst:.5f} ({l_inst - l_raw:+.5f}); "
          "the typical logit's ratio after/before per block: " + ' '.join(f"{r['logit_ratio_median']:.2f}" for r in recs))
model.train()
del x; torch.cuda.empty_cache()

# THE BENCH under the arm's precision: 3 micro-steps, the pace and the peak memory
opt_probe = torch.optim.SGD(model.parameters(), lr=0.0)
xb = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
for i in range(3):
    with autocast('cuda', tc.precision):
        out = model(xb, targets=xb)
    out.loss.backward(); opt_probe.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3
vram = torch.cuda.max_memory_allocated() / 2**30
toks = tc.micro_batch * cfg.context
print(f'bench ({tc.precision}): {sps:.2f}s / micro-step · {toks/sps/1e3:.0f}k tok/s · x{tc.grad_accum} accum = {sps*tc.grad_accum:.1f} s/step · peak {vram:.1f} GB')
assert vram < 88, 'over the card budget at this micro_batch — lower micro_batch and raise grad_accum (the step stays 262,144 tokens)'
del model, out, opt_probe, sd; torch.cuda.empty_cache()
print('PREFLIGHT PASS — C3 is armed')

In [ ]:
# C3 (T) — THE SESSION. The FIRST session starts WEIGHTS-ONLY (the trainer does it when this craft has no record on the hub:
# the source weights, fresh optimizers, the cursor at START_STEP, the fineweb stream at a fresh shuffle, the QK-norm gains
# installed from one fineweb batch — all recorded in the manifest); every later session resumes from this craft's own
# resume/latest.pt. Stops at phase boundaries; every boundary ships a checkpoint (the trainer) AND a report JSON (this cell).
# The phases are chronological (fineweb_main's remainder, S0-S8, anneal_nochat, anneal_mix): no anneal activation is needed.
import gc, json, time, torch
from google.colab import userdata
from geolip.alephllm import prepare
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, special_token_gauge
from geolip.alephllm.model.governor import govern_model

PRESET = craft(CRAFT, START_STEP)
MAX_HOURS = 10.5

run = prepare(PRESET, hf_token=userdata.get('HF_TOKEN'))
if run.manifest.init_from:
    start = dict(run.manifest.init_from); gains = start.pop('qk_gains', None)
    print('start:', json.dumps(start, indent=1))
    if gains:
        print('QK-norm gains (block: the typical logit\'s ratio after/before):',
              ' '.join(f"{r['block']}:{r['logit_ratio_median']:.2f}" for r in gains['blocks']))

def boundary_report(run, tag):
    m, tok, dev = run.raw_model, run.tokenizer, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, tok, dev)
        sample = torch.randint(0, 255, (2, min(1024, m.cfg.context)), device=dev)
        census = model_census(m, sample)
        ledger = toggle_ledger(m, run._val())
        sp = special_token_gauge(m, run._val())
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen,
           'craft': run.cfg.name, 'precision': run.tc.precision, 'qk_norm': run.cfg.qk_norm, 'attn_fp32': run.cfg.attn_fp32,
           'init_from': {k: v for k, v in (run.manifest.init_from or {}).items() if k != 'qk_gains'},
           'probes': pr, 'census_flags': census.get('flags'), 'ledger': ledger, 'special': sp,
           'governor_hits_cum': getattr(run, '_gov_hits', 0),
           'crowd_check_extra_hits': govern_model(m, run.tc.governor_theta) if run.tc.governor else None}
    run.hub.upload_bytes(json.dumps(rep, default=float).encode(), f'reports/resume/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

# the driver contract: train(stop_at_boundary=True) RETURNS at every phase boundary with run._last_boundary = the phase
# that just finished; run._interrupted = a manual stop (never auto-resumed).
t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < 0.2: break
    run.train(max_hours=hours_left, stop_at_boundary=True)
    if getattr(run, '_interrupted', False):
        print('manual stop - no auto-resume; resume state is on the hub')
        break
    tag = getattr(run, '_last_boundary', None) or 'session_cap'
    boundary_report(run, tag)
    if run.manifest.current_phase() is None:
        print('curriculum complete - the arm is FINISHED'); break
    gc.collect(); torch.cuda.empty_cache()
print('session over — resume state on the hub')

In [ ]:
# C4 — growth table: this arm's boundary reports beside the first run's (the same tags; the comparator).
import json
from huggingface_hub import HfApi, hf_hub_download
from geolip.alephllm.presets import TRAINING_REPO
PRESET = craft(CRAFT, START_STEP)
api = HfApi()
def rows_for(prefix, sub):
    files = sorted(f for f in api.list_repo_files(TRAINING_REPO) if f.startswith(f'{prefix}/{sub}/') and f.endswith('.json'))
    rows = []
    for f in files:
        r = json.load(open(hf_hub_download(TRAINING_REPO, f)))
        pr = r.get('probes')
        if not pr:
            continue
        accs = {k.split('_', 1)[0]: round(v.get('acc', v) if isinstance(v, dict) else v, 2) for k, v in pr.items()}
        led = r.get('ledger') or {}
        sp = r.get('special') or {}
        rows.append((r['tag'][:14], r['step'], round(r['tokens'] / 1e9, 2), led.get('bpb_full'),
                     led.get('toggle_head_aleph_off'), sp.get('doc_bpb'), accs))
    rows.sort(key=lambda x: x[1])
    return rows
def show(title, rows):
    print(title)
    print(f"{'stage':<15}{'step':>7}{'tok(B)':>7}{'val':>7}{'head':>8}{'doc':>6}  probes")
    for tag, step, tk, val, hd, doc, accs in rows:
        v = f'{val:.3f}' if isinstance(val, float) else '  --'
        h = f'{hd:+.3f}' if isinstance(hd, float) else '    --'
        d = f'{doc:.1f}' if isinstance(doc, float) else '  --'
        print(f'{tag:<15}{step:>7}{tk:>7}{v:>7}{h:>8}{d:>6}  {accs}')
show(f'== {PRESET.model.name} (this arm)', rows_for(PRESET.model.name, 'reports/resume'))
show(f"== {PRESET.init_from['prefix']} (the first run, the comparator)", rows_for(PRESET.init_from['prefix'], 'reports/v2'))

### Notes
- **What to watch** (TensorBoard under the craft's prefix): `train/grad_norm` against the clip (1.0). The first run's
  pre-clip norm crossed it at step 17,600 and its stage medians rose from 0.29 (fineweb) to 6.5 (S0) and past 100 later; its
  held-out loss on the fineweb gauge went 1.18 → 3.19 → 2.88. An arm that holds the norm under the clip through S0 and S1
  (the screen: 16,000 → 25,561) is the result; the `-bf16` arm is what the others are read against.
- **Provenance the manifest records**: the source checkpoint and step; the bf16 → fp32 upcast; fresh optimizer states; the
  cursor (3.894B of 5B fineweb tokens counted); the fineweb seed offset (7919); the QK-norm gain install (the batch's dataset
  and seed, per-block gains and the logit-scale ratio). Reports land under `reports/resume/`.
- **Sessions**: boundary-exact and interrupt-safe as before; a second session on the same arm is a normal resume.
- **Another start**: set `START_STEP = 14000` (the other intact checkpoint); the arm then runs under `<name>-r14k`.
- **Pace**: the full-fp32 attention arms (`-attn`, `-fix`) run the memory-efficient fp32 kernel, which has no fused form
  (7.2 s a step on an H100 at 8 × 8); the `-fp16` arms run fp16 flash inside the fp32 block at about the control's pace;
  the full-fp32 arms are the slowest. C2's bench line is the number to book by; C3's `[attn]` line names the kernel.